In [1]:
from nichenetpy.utils import (
    read_csv_cols,
    read_csv_rows,
    extract_ligands_from_settings
)
from nichenetpy.model_construction import (
    construct_weighted_networks,
    construct_ligand_target_matrix,
    apply_hub_correction
)
from nichenetpy.evaluation import (
    convert_expression_settings_evaluation,
    convert_settings_ligand_prediction,
    get_single_ligand_importances,
    evaluate_single_importances_ligand_prediction
)
from nichenetpy.prediction import LigandActivityPredictor

from itertools import chain, repeat

import os
import requests
import pandas as pd
import session_info
import json
import numpy as np

In [2]:
network_path = os.path.normpath("./tutorial_files/model_construction/human")
if not os.path.exists(network_path):
    os.makedirs(network_path)
for filename in (
    "gr_human.csv",
    "lr_network_human.csv",
    "lr_sig_human.csv",
    "optimized_source_weights.csv",
    "annotation_data_sources.csv"
):
    file_path = os.path.join(network_path, filename)
    if not os.path.exists(file_path):
        res = requests.get(f"https://zenodo.org/records/14929618/files/{filename}")
        with open(file_path, "wb") as file:
            file.write(res.content)

In [3]:
gr_network = pd.DataFrame(read_csv_cols(os.path.join(network_path, "gr_human.csv")))
lr_network = pd.DataFrame(read_csv_cols(os.path.join(network_path, "lr_network_human.csv")))
sig_network = pd.DataFrame(read_csv_cols(os.path.join(network_path, "lr_sig_human.csv")))

In [4]:
train_path = "D:/Data/nichenetpy/model_optimization"
with open(os.path.join(train_path, "settings_training_f1234.json"), "rb") as file:
    settings_CV = json.loads(file.read())
settings = settings_CV["settings"]
ligands = extract_ligands_from_settings(settings)

In [5]:
gr_network = gr_network[
    ((gr_network["database"] == "NicheNet_LT") & np.array([fr not in settings_CV["forbidden_ligands_nichenet"] for fr in gr_network["from"]]))
    |
    ((gr_network["database"] == "CytoSig") & np.array([fr not in settings_CV["forbidden_ligands_cytosig"] for fr in gr_network["from"]]))
]

In [6]:
source_weights = dict(zip(set(chain(gr_network["source"], lr_network["source"], sig_network["source"])), repeat(1)))
weighted_networks = construct_weighted_networks(
    lr_network,
    sig_network,
    gr_network,
    source_weights
)
weighted_networks["lr_sig"] = apply_hub_correction(weighted_networks["lr_sig"], hub=0.115)
weighted_networks["gr"] = apply_hub_correction(weighted_networks["gr"], hub=0.0803)

In [7]:
predictor = LigandActivityPredictor(
    *construct_ligand_target_matrix(
        weighted_networks,
        lr_network,
        ligands,
        damping_factor=0.789,
        ltf_cutoff=0.926
    )
)
predictor.replace_zero_col_by_noisy_scores()

In [8]:
performances = {
    k: predictor.evaluate_target_prediction(v["from"] if type(v["from"]) is str else "-".join(v["from"]), v["response"])
    for k, v in settings.items()
}

In [34]:
all_ligands = extract_ligands_from_settings(settings, combination=False)
ligand_importances = {
    "setting_id": [],
    "test_ligand": [],
    "true_ligand": [],
    "auroc": [],
    "pearson": [],
    "aupr": [],
    "aupr_corrected": []
}
for setting_id, setting in list(settings.items())[:3]:
    for ligand in all_ligands:
        ligand_importances["setting_id"].append(setting_id)
        ligand_importances["test_ligand"].append(ligand)
        ligand_importances["true_ligand"].append(setting["from"])
        for k, v in predictor.evaluate_target_prediction(ligand, setting["response"]).items():
            ligand_importances[k].append(v)
ligand_importances = pd.DataFrame(ligand_importances)

In [35]:
ligand_importances

,setting_id,test_ligand,true_ligand,auroc,pearson,aupr,aupr_corrected
0,ifna_ifng_timeseries,LTB,IFNG,0.525002,-0.007439,0.010134,0.002284
1,ifna_ifng_timeseries,IL12B,IFNG,0.506631,0.002104,0.008492,0.000643
2,ifna_ifng_timeseries,FGF7,IFNG,0.632259,0.153786,0.058166,0.050316
3,ifna_ifng_timeseries,INHBA,IFNG,0.509115,0.002717,0.008072,0.000222
4,ifna_ifng_timeseries,HMGB1,IFNG,0.517136,0.005140,0.008109,0.000259
...,...,...,...,...,...,...,...
184,GSE6085_IL2_timeseries,WNT1,IL2,0.662281,0.220015,0.113805,0.087579
185,GSE6085_IL2_timeseries,VEGFA,IL2,0.504340,0.002418,0.026704,0.000478
186,GSE6085_IL2_timeseries,BMP2,IL2,0.487868,-0.006683,0.025582,-0.000644
187,GSE6085_IL2_timeseries,IL1A,IL2,0.496757,-0.001803,0.026347,0.000120


In [10]:
session_info.show()